In [1]:
import pandas as pd

file1 = './data/apt_seoul.xlsx'
file2 = './data/apt_seoul_22_23.xlsx'

df1 = pd.read_excel(file1, skiprows=12, thousands=',')
df2 = pd.read_excel(file2, skiprows=12, thousands=',')

C:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\User\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [2]:
print(df1.shape)
print(df2.shape)

(76666, 21)
(16925, 21)


In [4]:
def apt_preprocess(df):
    # 불편한 칼럼이름 변경 - 전용면적(㎡)	,거래금액(만원)
    # df.rename() 바뀐 복사본 반환 . 원본이 안바뀜
    # 1. df = df.rename(....) #복사본을 원본에 저장
    # 2. df.rename(...  ,inplace=True) #원본을 수정
    df.rename( 
        columns={
            "전용면적(㎡)":"전용면적", 
            "거래금액(만원)":"거래금액"
        }, inplace=True
    )
    
    # 시군구를 구,동으로 분리저장
    df['구'] = df['시군구'].str.split().str[1]
    df['동'] = df['시군구'].str.split().str[2]
    
    # 전용면적 -> 평
    df['평'] = df['전용면적'] / 3.3
    
    # 평 -> 평형
    def to_ph(x):
        if x < 10: return '10평이하'
        if x < 20: return '10평대'
        if x < 30: return '20평대'
        if x < 40: return '30평대'
        return '40평이상'
    df['평형'] = df['평'].apply(to_ph)
    
    # 거래취소된 행 삭제(정상거래된 행)
    df = df.query(' 해제사유발생일 == "-" ')
    
    # 가격에 영향없는 칼럼 삭제
    df.drop(columns=['NO', '번지', '본번', '부번','매수자', '매도자', '도로명', '해제사유발생일', '거래유형', '중개사소재지',
           '등기일자', '주택유형',], inplace=True)
    
    # 거래금액을 억단위로 변환
    df['거래금액'] = df['거래금액'] / 10000
    
    # 계약년월+계약일 -> 요일
    df['계약일자'] = pd.to_datetime(
        df['계약년월'].astype(str) + df['계약일'].astype(str).str.zfill(2),
        format='%Y%m%d'
    )
    
    df['계약요일'] = df['계약일자'].dt.day_name().map({
        'Monday': '월', 'Tuesday': '화', 'Wednesday': '수',
        'Thursday': '목', 'Friday': '금', 'Saturday': '토', 'Sunday': '일'
    })
    return df

In [5]:
df1 = apt_preprocess(df1)
df1.head()

C:\Users\User\AppData\Local\Temp\ipykernel_6680\2702626630.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['NO', '번지', '본번', '부번','매수자', '매도자', '도로명', '해제사유발생일', '거래유형', '중개사소재지',
C:\Users\User\AppData\Local\Temp\ipykernel_6680\2702626630.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['거래금액'] = df['거래금액'] / 10000
C:\Users\User\AppData\Local\Temp\ipykernel_6680\2702626630.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the 

,시군구,단지명,전용면적,계약년월,계약일,거래금액,동,층,건축년도,구,평,평형,계약일자,계약요일
0,서울특별시 은평구 수색동,DMC 진흥,78.2853,202604,4,7.75,수색동,15,2003,은평구,23.722818,20평대,2026-04-04,토
1,서울특별시 서대문구 북가좌동,DMC래미안e편한세상,153.8600,202604,3,16.50,북가좌동,8,2012,서대문구,46.624242,40평이상,2026-04-03,금
2,서울특별시 서대문구 대현동,신촌럭키,59.7000,202604,3,12.00,대현동,13,1999,서대문구,18.090909,10평대,2026-04-03,금
3,서울특별시 성동구 성수동1가,동아그린,58.3200,202604,3,13.50,성수동1가,12,1999,성동구,17.672727,10평대,2026-04-03,금
4,서울특별시 송파구 잠실동,리센츠,84.9900,202604,3,35.00,잠실동,11,2008,송파구,25.754545,20평대,2026-04-03,금


In [6]:
df2 = apt_preprocess(df2)
df2.head()

C:\Users\User\AppData\Local\Temp\ipykernel_6680\2702626630.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['NO', '번지', '본번', '부번','매수자', '매도자', '도로명', '해제사유발생일', '거래유형', '중개사소재지',
C:\Users\User\AppData\Local\Temp\ipykernel_6680\2702626630.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['거래금액'] = df['거래금액'] / 10000
C:\Users\User\AppData\Local\Temp\ipykernel_6680\2702626630.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the 

,시군구,단지명,전용면적,계약년월,계약일,거래금액,동,층,건축년도,구,평,평형,계약일자,계약요일
0,서울특별시 동대문구 답십리동,청계푸르지오시티,30.420,202304,10,3.57,답십리동,7,2015,동대문구,9.218182,10평이하,2023-04-10,월
1,서울특별시 중랑구 상봉동,동일스카이시티102동,63.820,202304,10,4.30,상봉동,4,2010,중랑구,19.339394,10평대,2023-04-10,월
2,서울특별시 노원구 중계동,동진신안,101.990,202304,10,9.50,중계동,1,1993,노원구,30.906061,30평대,2023-04-10,월
3,서울특별시 노원구 공릉동,풍림아파트A,84.990,202304,10,7.53,공릉동,2,2001,노원구,25.754545,20평대,2023-04-10,월
4,서울특별시 동작구 상도동,포스코더샵,132.735,202304,10,14.10,상도동,12,2007,동작구,40.222727,40평이상,2023-04-10,월


### 3년전에 비해 거래금액이 가장 많이 오른 구
- 구별 거래금액 평균

In [12]:
# 현재 구별 거래금액 평균
avg_price_1 = df1.groupby('구')['거래금액'].mean().to_frame()
avg_price_1.columns=["현거래금액"]
avg_price_2 = df2.groupby('구')['거래금액'].mean().to_frame()
avg_price_2.columns=["3년전거래금액"]

In [15]:
#가로로 연결
price_all = pd.concat([avg_price_1, avg_price_2], axis=1)
price_all

,현거래금액,3년전거래금액
구,,
강남구,28.351920,19.309195
강동구,12.682820,9.425547
강북구,6.588776,5.421509
강서구,8.877549,6.948915
관악구,8.337241,6.418198
광진구,14.191279,9.996963
구로구,7.135536,5.399239
금천구,6.419462,4.845835
노원구,6.348460,6.099202


In [21]:
# 현거래금액, 3년전거래금액간 차이는 diff칼럼에 저장
price_all['diff'] = price_all['현거래금액'] - price_all['3년전거래금액'] 
price_all.sort_values('diff', ascending=False)

,현거래금액,3년전거래금액,diff
구,,,
강남구,28.351920,19.309195,9.042725
동작구,13.177472,8.250724,4.926748
영등포구,12.667759,8.458512,4.209247
광진구,14.191279,9.996963,4.194316
마포구,14.777156,10.773008,4.004148
종로구,11.675781,7.883821,3.791960
양천구,13.296219,9.938863,3.357355
송파구,18.683833,15.333615,3.350219
강동구,12.682820,9.425547,3.257273


### 평형별 3년전 거래금액과의 차이

In [23]:
# 현재 구별 거래금액 평균
avg_price_1 = df1.groupby('평형')['거래금액'].mean().to_frame()
avg_price_1.columns=["현거래금액"]
avg_price_2 = df2.groupby('평형')['거래금액'].mean().to_frame()
avg_price_2.columns=["3년전거래금액"]

#가로로 연결
price_all = pd.concat([avg_price_1, avg_price_2], axis=1)
price_all

# 현거래금액, 3년전거래금액간 차이는 diff칼럼에 저장
price_all['diff'] = price_all['현거래금액'] - price_all['3년전거래금액'] 
price_all.sort_values('diff', ascending=False)

,현거래금액,3년전거래금액,diff
평형,,,
10평대,9.057305,7.386134,1.671172
20평대,12.950721,11.497260,1.453460
30평대,17.312064,16.504605,0.807458
10평이하,2.951654,2.204360,0.747293
40평이상,28.708852,27.962658,0.746194
